# Docking Demo: AutoDock Vina on TBXT Pocket A

Single-compound docking with multi-conformer scoring, interaction analysis,
and 3D visualization.

**Setup:** This notebook requires the `tbxt-dock` conda environment:
```bash
conda activate tbxt-dock
jupyter lab
```

In [1]:
from tbxt_hackathon.docking import dock_single, dock_batch

## 1. Single compound docking

Dock a difluorobenzamide (similar to the reference ligand in the pocket PDB)
using 3 independent conformer runs, exhaustiveness=16, 5 poses each.

In [2]:
POCKET_PDB = "../data/structures/TGT_TBXT_A_pocket.pdb"
SMILES = "COC(=O)c1ccccc1C1CCN(C(=O)Nc2cc(NC(C)=O)ccc2F)CC1"

report = dock_single(
    SMILES,
    POCKET_PDB,
    n_conformers=3,
    exhaustiveness=16,
    n_poses=5,
)
print(report.summary())

Docking report: COC(=O)c1ccccc1C1CCN(C(=O)Nc2cc(NC(C)=O)ccc2F)CC1
  Best score:        -5.17 kcal/mol  (conformer 0, pose 0)
  Inter-molecular:   -6.68 kcal/mol
  Intra-molecular:   -0.96 kcal/mol
  Ligand efficiency: 0.172 kcal/mol/HA
  Heavy atoms:       30
  MW:                413.4
  Conformers run:    3
  Total poses:       15
  Score range:       -5.17 to -4.37 kcal/mol

  Best-pose interactions (7):
    hydrophobic    LEU61    CD2   3.57 A
    hydrophobic    VAL93    CG2   3.78 A
    hydrophobic    TYR94    CA    3.74 A
    hydrophobic    ILE95    CD1   3.56 A
    hydrophobic    GLY126   CA    3.70 A
    hydrophobic    HIS141   CD2   3.79 A
    hydrophobic    ILE152   CD1   3.57 A


## 2. Explore the report

In [3]:
import pandas as pd

pose_df = pd.DataFrame([
    {
        "idx": i,
        "conformer": p.conformer_idx,
        "pose": p.pose_idx,
        "score": p.score,
        "inter": p.inter_energy,
        "intra": p.intra_energy,
        "n_interactions": len(p.interactions),
        "hbonds": sum(1 for x in p.interactions if x.type == "hbond"),
        "pi_stack": sum(1 for x in p.interactions if x.type == "pi_stacking"),
        "hydrophobic": sum(1 for x in p.interactions if x.type == "hydrophobic"),
        "salt_bridge": sum(1 for x in p.interactions if x.type == "salt_bridge"),
    }
    for i, p in enumerate(report.poses)
])
pose_df.style.background_gradient(subset=["score"], cmap="RdYlGn_r")

,idx,conformer,pose,score,inter,intra,n_interactions,hbonds,pi_stack,hydrophobic,salt_bridge
0,0,0,0,-5.171000,-6.682000,-0.959000,7,0,0,7,0
1,1,0,1,-5.043000,-6.628000,-0.848000,7,1,0,6,0
2,2,0,2,-4.995000,-6.285000,-1.129000,10,3,1,6,0
3,3,0,3,-4.599000,-5.835000,-1.068000,6,0,1,5,0
4,4,0,4,-4.574000,-5.883000,-0.988000,6,0,1,5,0
5,5,1,0,-5.065000,-6.545000,-0.930000,7,0,0,7,0
6,6,1,1,-5.046000,-6.494000,-0.957000,6,1,0,5,0
7,7,1,2,-4.964000,-6.198000,-1.148000,10,3,1,6,0
8,8,1,3,-4.927000,-6.251000,-1.046000,6,0,1,5,0
9,9,1,4,-4.739000,-6.103000,-0.952000,8,0,1,7,0


## 3. Best pose: 3D visualization with interactions

Gold = H-bond, green = pi-stacking, gray = hydrophobic, red = salt bridge.

In [4]:
report.show_3d()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 4. Step through individual poses

Use `report.show_3d(pose_idx=N)` to inspect any pose.
The table above shows the index to use.

In [5]:
# Show the second-best pose (change the index to explore others)
second_best_idx = pose_df.sort_values("score").iloc[1]["idx"]
print(report.poses[int(second_best_idx)].summary())
report.show_3d(pose_idx=int(second_best_idx))

  Conformer 1, pose 0:
    Score:         -5.07 kcal/mol
    Inter:         -6.54 kcal/mol
    Intra:         -0.93 kcal/mol
    Interactions:  7
      hydrophobic    SER59    CB    3.73 A
      hydrophobic    LEU61    CD2   3.61 A
      hydrophobic    PRO90    CB    3.82 A
      hydrophobic    CYS92    CA    3.84 A
      hydrophobic    VAL93    CB    3.75 A
      hydrophobic    ILE95    CD1   3.69 A
      hydrophobic    VAL143   CG1   3.45 A


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 5. Overlay all poses

In [6]:
report.show_all_poses()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 6. Best-pose interaction details

In [7]:
bp = report.best_pose
ix_df = pd.DataFrame([
    {
        "type": ix.type,
        "residue": ix.protein_residue,
        "protein_atom": ix.protein_atom,
        "ligand_atom_idx": ix.ligand_atom_idx,
        "distance_A": ix.distance,
        "angle": ix.angle,
        "detail": ix.detail,
    }
    for ix in bp.interactions
])
ix_df

,type,residue,protein_atom,ligand_atom_idx,distance_A,angle,detail
0,hydrophobic,LEU61,CD2,12,3.57,None,
1,hydrophobic,VAL93,CG2,0,3.78,None,
2,hydrophobic,TYR94,CA,22,3.74,None,
3,hydrophobic,ILE95,CD1,18,3.56,None,
4,hydrophobic,GLY126,CA,22,3.70,None,
5,hydrophobic,HIS141,CD2,0,3.79,None,
6,hydrophobic,ILE152,CD1,5,3.57,None,
